# Society of Mind

In [ ]:
print("\n🏗️  Society of Mind Architecture:")
print("""
    ┌─────────────────────────────────────-┐
    │        Society of Mind Agent         │
    │  ┌─────────────────────────────────┐ │
    │  │         Inner Team              │ │
    │  │  ┌─────────┐  ┌─────────────┐   │ │
    │  │  │ Agent A │  │   Agent B   │   │ │
    │  │  │(Writer) │  │  (Editor)   │   │ │
    │  │  └─────────┘  └─────────────┘   │ │
    │  │         │            │          │ │
    │  │         └────┬───────┘          │ │
    │  │              │                  │ │
    │  │         Discussion              │ │
    │  └─────────────────────────────────┘ │
    │                 │                    │
    │            Synthesis                 │
    │                 │                    │
    │           Final Response             │
    └─────────────────────────────────────-┘
    """)

In [ ]:
api_key=None

In [ ]:
import asyncio
from autogen_agentchat.ui import Console
from autogen_agentchat.agents import AssistantAgent, SocietyOfMindAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination


async def main() -> None:
    model_client = OpenAIChatCompletionClient(model="gpt-4o",api_key=api_key)

    agent1 = AssistantAgent("assistant1", model_client=model_client, system_message="You are a writer, write well under 50 words.")
    agent2 = AssistantAgent(
        "assistant2",
        model_client=model_client,
        system_message="You are an editor, provide critical feedback under 50 words. Respond with 'APPROVE' if the text addresses all feedbacks.",
    )
    inner_termination = TextMentionTermination("APPROVE")
    inner_team = RoundRobinGroupChat([agent1, agent2], termination_condition=inner_termination)


    society_of_mind_agent = SocietyOfMindAgent("society_of_mind", team=inner_team, model_client=model_client,response_prompt='Output a standalone response to the original request under 50 words, without mentioning any of the intermediate discussion.')

    agent3 = AssistantAgent(
        "assistant3", model_client=model_client, system_message="Translate the text to Spanish under 50 words."
    )
    team = RoundRobinGroupChat([society_of_mind_agent, agent3], max_turns=2)

    stream = team.run_stream(task="Write a short story with a surprising ending under 50 words.")
    await Console(stream)


await (main())


---------- TextMessage (user) ----------
Write a short story with a surprising ending under 50 words.
Write a short story with a surprising ending under 50 words.
---------- TextMessage (assistant1) ----------
Lila cherished a dusty lamp she found, dreaming of magic. One evening, she rubbed it, wishing for adventure. To her surprise, a butterfly emerged, transforming her small room into a vibrant jungle. Heart pounding, she realized the greatest magic: the power of imagination.
---------- TextMessage (assistant2) ----------
Consider clarifying how the butterfly leads to the jungle transformation, as the connection may be vague to readers. The reveal of imagination's power is effective, but visually depicting the change can enhance surprise. Otherwise, the twist is delightful.
---------- TextMessage (assistant1) ----------
Lila cherished a dusty lamp, dreaming of magic. One evening, she rubbed it, wishing for adventure. To her surprise, a butterfly emerged, fluttering around, transformi